# 전체 노트북 초기화 검사

별도 검증 폴더에서 각 노트북의 첫 코드 셀만 실행해 루트 탐색·입력 읽기·환경 로딩을 점검합니다. 분석 전체 실행 검증은 `verify_reproduction.ipynb`에서 별도로 수행합니다. 네트워크 요청은 차단합니다.

In [ ]:
from pathlib import Path
import os, json, shutil, io, contextlib, traceback
from unittest.mock import patch
import nbformat
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'environment/path_aliases.json').is_file())
WORK = ROOT/'environment/initialization-workspace'
WORK.mkdir(exist_ok=True)
for folder in ['jupyter-notebook','reports','metadata','documents','dataset/analysis','dataset/processed']:
    shutil.copytree(ROOT/folder, WORK/folder, dirs_exist_ok=True)
for name in ['AGENTS.md','README.md','requirements.txt']:
    shutil.copy2(ROOT/name, WORK/name)
(WORK/'environment').mkdir(exist_ok=True)
for name in ['project_paths.ipynb','path_aliases.json','path_migration.json']:
    shutil.copy2(ROOT/'environment'/name, WORK/'environment'/name)
if not (WORK/'dataset/raw').exists():
    import subprocess
    subprocess.run(['powershell','-NoProfile','-Command',
        "New-Item -ItemType Junction -Path '"+str(WORK/'dataset/raw').replace("'","''")+
        "' -Target '"+str(ROOT/'dataset/raw').replace("'","''")+"' | Out-Null"],check=True,capture_output=True)
records = []


## 초기화 실행

각 노트북은 독립된 변수 공간에서 실행합니다. 실행 진입점의 첫 셀이 만드는 실행 기록은 노트북마다 원본으로 복원하며, 수집 요청은 허용하지 않습니다.

In [ ]:
def denied(*args, **kwargs):
    raise AssertionError('초기화 검사 중 네트워크 요청은 허용하지 않습니다.')

for original in sorted((ROOT/'jupyter-notebook').rglob('*.ipynb')):
    relative = original.relative_to(ROOT)
    path = WORK/relative
    for folder in ['reports','jupyter-notebook/sandbox/reports']:
        shutil.copytree(ROOT/folder, WORK/folder, dirs_exist_ok=True)
    notebook = nbformat.read(path,as_version=4)
    first = next(cell.source for cell in notebook.cells if cell.cell_type=='code')
    scope = {'__name__':'__notebook_initialization__'}
    try:
        os.chdir(path.parent)
        with patch('urllib.request.OpenerDirector.open',denied), patch('requests.sessions.Session.request',denied), contextlib.redirect_stdout(io.StringIO()):
            exec(compile(first,str(relative),'exec'),scope)
        assert scope['ROOT'] == WORK, (scope['ROOT'], WORK)
        records.append({'notebook':relative.as_posix(),'status':'passed'})
    except Exception as exc:
        records.append({'notebook':relative.as_posix(),'status':'failed','error':str(exc),'traceback':traceback.format_exc()})
    finally:
        os.chdir(ROOT)
output = ROOT/'environment/verification-output/initialization_checks.json'
output.parent.mkdir(exist_ok=True)
output.write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(records)[['notebook','status']])
assert all(item['status']=='passed' for item in records), [item for item in records if item['status']=='failed']


## 해석

통과한 노트북은 현재 구조에서 초기 입력과 환경을 찾았습니다. 이 검사는 전체 통계 계산 검증을 대신하지 않습니다.

### 2026-10-06 검증 기록

전체 56개 노트북 초기화 통과. 경로 전수 문법 검사 56개, 실행 진입점 4개, 결과표 8개 수치 재현과 추가 보고서 7개 실행 검증도 통과했습니다. 상세 기록은 `verification-output/`에 있습니다.